<a href="https://colab.research.google.com/github/Nasim-Amani/grid-failure/blob/main/NaNs_(mount_XLSX).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Summary — Categorical Missingness & Network Feature Audit

We investigated the missingness, recoverability, temporal stability, and category consistency of the two most important network categorical features: `zone` and `lv_feeder`.

### 1. `zone` — ❌ Exclude as a primary predictor

- Missing in **48.70%** of the 2,764 event records.
- Completely missing during **1395–1398**, indicating a clear change in the data-recording process.
- Its observed categories vary substantially across years and source files (**224 categories overall**).
- MV feeder information cannot deterministically recover it.
- Distribution/GIS information recovers only about **1.63%** of missing values.
- Therefore, imputing missing `zone` values would introduce unreliable information.

**Final treatment:** Exclude `zone` from the main model. If retained for a sensitivity analysis, represent missing values explicitly as `Unknown` rather than imputing them.

---

### 2. `lv_feeder` — ✅ Retain

- Missing in **28.73%** of event records.
- Present throughout all **10 years** of the dataset.
- Feeder IDs **1–10 occur in every year**, while 11–12 recur across most years.
- The feature has a limited and manageable number of legitimate categories (**16 IDs: 1–15 and 21**).
- Its category structure is substantially more stable than `zone`.
- Network identifiers can recover only a small portion of missing values (maximum about **13.73%**), so blind imputation is not justified.

### Exceptional values

The values `0`, `9999`, and `101782` occur only **5 records in total** and behave as anomalies rather than consistent feeder IDs.

**Final treatment:**

| `lv_feeder` value | Treatment |
|---|---|
| 1–15, 21 | Keep as categorical feeder IDs |
| 0 | Convert to missing |
| 9999 | Convert to missing |
| 101782 | Convert to missing |
| Missing | Keep as missing/Unknown |

---

## Final Modeling Decision

`zone` → **Exclude from the main model.**

`lv_feeder` → **Keep as a nominal categorical network identifier.**

Do **not** treat `lv_feeder` as a numerical variable because feeder ID 10 is not mathematically greater than feeder ID 2.

Do **not** blindly impute missing `lv_feeder` values because the available network fields cannot reliably reconstruct them.

For encoding, a **native categorical approach such as CatBoost** is preferred over automatically creating a large one-hot representation.

### Overall conclusion

The audit shows that `zone` suffers from a **structural/data-collection problem**, whereas `lv_feeder` represents a **usable and relatively stable network feature with genuine missing values and a few anomalous codes**.

Therefore, the final dataset should **drop `zone` from the primary modeling feature set, clean and retain `lv_feeder`, and handle its missingness explicitly rather than fabricating feeder identities.**

In [6]:
# ==============================
# 1. Mount Google Drive
# ==============================

from google.colab import drive

drive.mount('/content/drive')

# ==============================
# 2. Import libraries
# ==============================

import os
import zipfile
import pandas as pd
import glob


# ==============================
# 3. Define paths
# ==============================

# Change this to your ZIP file location in Google Drive
zip_path = "/content/drive/MyDrive/feeders_DATA.zip"

# Where to extract the ZIP file
extract_path = "/content/extracted_dataset"


# ==============================
# 4. Extract ZIP file
# ==============================

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted successfully!")


# ==============================
# 5. Find all XLSX files
# ==============================

xlsx_files = glob.glob(
    os.path.join(extract_path, "**/*.xlsx"),
    recursive=True
)

print(f"Number of Excel files found: {len(xlsx_files)}")

for file in xlsx_files:
    print(file)


# ==============================
# 6. Read all Excel files
# ==============================

dataframes = []

for file in xlsx_files:
    print(f"Reading: {file}")

    df = pd.read_excel(
        file,
        sheet_name=0   # read first sheet only
    )

    # Keep track of source file
    df["source_file"] = os.path.basename(file)

    dataframes.append(df)


# ==============================
# 7. Combine all files
# ==============================

data = pd.concat(
    dataframes,
    ignore_index=True
)


# ==============================
# 8. Basic inspection
# ==============================

print("\nDataset shape:")
print(data.shape)

print("\nFirst 5 rows:")
display(data.head())

print("\nColumn names:")
print(data.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP extracted successfully!
Number of Excel files found: 19
/content/extracted_dataset/DATA/1395-2.xlsx
/content/extracted_dataset/DATA/1404-1-.xlsx
/content/extracted_dataset/DATA/1398-2.xlsx
/content/extracted_dataset/DATA/1399-1.xlsx
/content/extracted_dataset/DATA/1400-2.xlsx
/content/extracted_dataset/DATA/1400-1.xlsx
/content/extracted_dataset/DATA/1403-1.xlsx
/content/extracted_dataset/DATA/1398-1.xlsx
/content/extracted_dataset/DATA/1401-2.xlsx
/content/extracted_dataset/DATA/1396-1.xlsx
/content/extracted_dataset/DATA/1397-1.xlsx
/content/extracted_dataset/DATA/1395-1.xlsx
/content/extracted_dataset/DATA/1399-2.xlsx
/content/extracted_dataset/DATA/1401-1.xlsx
/content/extracted_dataset/DATA/1397-2.xlsx
/content/extracted_dataset/DATA/1402-2.xlsx
/content/extracted_dataset/DATA/1402-1.xlsx
/content/extracted_dataset/DATA/1396-2.xlsx
/content/extracted

,ردیف,روز هفته,ردیف یکتا,معاونت,منطقه اصلی,تاریخ اطلاع,ساعت اطلاع,پست فوق توزیع,کد پست فوق توزیع,فیدر فشار متوسط,...,گزارش کار اکیپ,اجازه کار,موقعیت خطا,شرح موقعیت خطا,کاربر ثبت کننده,ترانس MV,ترانس LV,GISCODE پست توزیع,GISCODE فیدر فشار متوسط,source_file
0,458,شنبه,5133814,معاونت شمالشرق,پاسداران,1395/12/28,14:10:00,شريعتي,257.0,MV_شريعتي_شهرزاد,...,NaN,NaN,NaN,NaN,حجت مومني,NaN,NaN,E914@GP13080000C04E,112280,1395-2.xlsx
1,1080,پنجشنبه,7445411,معاونت شمالشرق,نیاوران,1395/12/26,09:50:00,شريعتي,257.0,MV_شريعتي_زمرد,...,NaN,NaN,NaN,NaN,حجت مومني,NaN,NaN,E914@GP13130004ECBE,112240,1395-2.xlsx
2,3675,چهارشنبه,2990116,معاونت شمالشرق,پاسداران,1395/12/18,05:29:00,شريعتي,257.0,MV_شريعتي_بيات,...,NaN,NaN,NaN,NaN,حجت مومني,NaN,NaN,E914@GP13080000C03E,112320,1395-2.xlsx
3,4048,سه شنبه,4355537,معاونت شمالشرق,پاسداران,1395/12/17,08:22:00,شريعتي,257.0,MV_شريعتي_حسينيه ارشاد,...,NaN,NaN,NaN,NaN,حجت مومني,NaN,NaN,E914@00000SB40173,112260,1395-2.xlsx
4,4340,دوشنبه,6915061,معاونت شمالشرق,پاسداران,1395/12/16,11:30:00,شريعتي,257.0,MV_شريعتي_بيات,...,NaN,NaN,NaN,NaN,حجت مومني,NaN,NaN,E914@QP13090009A7D3,112320,1395-2.xlsx



Column names:
['ردیف', 'روز هفته', 'ردیف یکتا', 'معاونت', 'منطقه اصلی', 'تاریخ اطلاع', 'ساعت اطلاع', 'پست فوق توزیع ', 'کد پست فوق توزیع', 'فیدر فشار متوسط', 'کد فیدر فشار متوسط', 'پست توزیع', 'کد پست توزیع', 'فیدر فشار ضعیف', 'آدرس', ' سطح ولتاژ', 'گروه قطع', 'علت قطع', 'اشکال', 'علت اشکال', 'جریان قطع اولیه', 'بار قطع شده (MWH)', 'انرژی توزیع نشده (KWH)', 'مدت قطع', ' جداکننده', 'نوع قطعی', 'وضعیت محل', 'مامور حوادث1', 'نوع بابرنامه', 'تکی یا کلی به تشخیص اپراتور', 'مامورحوادث2', 'زون ', 'منطقه 2', 'شرح حادثه', 'اطلاعات', 'زمان قطع', 'زمان وصل', 'مدت زمان خاموشی به دقیقه', 'كاربري كه ثبت كرده', 'نوع پست', 'نام پیمانکار', 'تلفن پیمانکار', 'مجموع تعداد کلیدزنی', 'گزارش دهنده', 'نوع سایر', 'گزارش کار اکیپ', 'اجازه کار', 'موقعیت خطا', 'شرح موقعیت خطا', 'کاربر ثبت کننده', 'ترانس MV', 'ترانس LV', 'GISCODE پست توزیع', 'GISCODE فیدر فشار متوسط', 'source_file']


In [7]:
# ============================================================
# ORIGINAL DATASET: CATEGORICAL MISSINGNESS ANALYSIS
# Purpose:
# Distinguish systematic/structural missingness from
# potentially accidental data-quality missingness.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Identify categorical columns
# ------------------------------------------------------------

categorical_cols = [
    col for col in data.columns
    if data[col].dtype == "object"
]

# Source file is provenance, not a categorical feature
categorical_cols = [
    col for col in categorical_cols
    if col != "source_file"
]

print("=" * 75)
print("ORIGINAL DATASET — CATEGORICAL MISSINGNESS AUDIT")
print("=" * 75)

print(f"\nDataset shape: {data.shape}")
print(f"Categorical/object columns: {len(categorical_cols)}")
print("\nCategorical columns:")
for col in categorical_cols:
    print(f"  • {col}")


# ------------------------------------------------------------
# 2. Overall missingness
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "column": categorical_cols,
    "missing_count": [
        data[col].isna().sum()
        for col in categorical_cols
    ],
    "missing_percent": [
        data[col].isna().mean() * 100
        for col in categorical_cols
    ],
    "observed_count": [
        data[col].notna().sum()
        for col in categorical_cols
    ],
    "unique_observed": [
        data[col].nunique(dropna=True)
        for col in categorical_cols
    ]
})

missing_summary = (
    missing_summary
    .sort_values("missing_percent", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 75)
print("1. OVERALL CATEGORICAL MISSINGNESS")
print("=" * 75)

display(
    missing_summary.style.format({
        "missing_percent": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 3. Missingness by feeder
# ------------------------------------------------------------

feeder_col = None

for candidate in [
    "فیدر فشار متوسط",
    "GISCODE فیدر فشار متوسط"
]:
    if candidate in data.columns:
        feeder_col = candidate
        break

if feeder_col is not None:

    feeder_missing = []

    for feeder, group in data.groupby(feeder_col, dropna=False):

        for col in categorical_cols:

            feeder_missing.append({
                "feeder": feeder,
                "column": col,
                "rows": len(group),
                "missing": group[col].isna().sum(),
                "missing_percent":
                    group[col].isna().mean() * 100
            })

    feeder_missing = pd.DataFrame(feeder_missing)

    print("\n" + "=" * 75)
    print("2. MISSINGNESS BY MV FEEDER")
    print("=" * 75)

    display(
        feeder_missing
        .sort_values(
            ["column", "missing_percent"],
            ascending=[True, False]
        )
        .style.format({
            "missing_percent": "{:.2f}%"
        })
    )

else:
    print("\nFeeder column not found; feeder-level analysis skipped.")


# ------------------------------------------------------------
# 4. Missingness by outage type
# ------------------------------------------------------------

outage_type_col = "نوع قطعی"

if outage_type_col in data.columns:

    print("\n" + "=" * 75)
    print("3. MISSINGNESS BY OUTAGE TYPE")
    print("=" * 75)

    print("\nOutage-type distribution:")
    display(
        data[outage_type_col]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    outage_missing = []

    for outage_type, group in data.groupby(
        outage_type_col,
        dropna=False
    ):

        for col in categorical_cols:

            outage_missing.append({
                "outage_type": outage_type,
                "column": col,
                "rows": len(group),
                "missing": group[col].isna().sum(),
                "missing_percent":
                    group[col].isna().mean() * 100
            })

    outage_missing = pd.DataFrame(outage_missing)

    display(
        outage_missing
        .sort_values(
            ["column", "missing_percent"],
            ascending=[True, False]
        )
        .style.format({
            "missing_percent": "{:.2f}%"
        })
    )

else:
    print(
        f"\n'{outage_type_col}' not found; "
        "outage-type analysis skipped."
    )


# ------------------------------------------------------------
# 5. Missingness co-occurrence
# ------------------------------------------------------------
# This tells us whether categorical fields disappear together.
# Strong co-missingness suggests systematic missingness.

print("\n" + "=" * 75)
print("4. MISSINGNESS CO-OCCURRENCE")
print("=" * 75)

missing_indicator = (
    data[categorical_cols]
    .isna()
    .astype(int)
)

co_missing = (
    missing_indicator
    .T
    .dot(missing_indicator)
)

print(
    "\nNumber of rows where BOTH categorical fields are missing:"
)

display(co_missing)


# ------------------------------------------------------------
# 6. Network-information completeness patterns
# ------------------------------------------------------------

network_categorical_cols = [
    col for col in [
        "منطقه اصلی",
        "زون",
        "پست توزیع",
        "فیدر فشار ضعیف",
        "سطح ولتاژ",
        "نوع پست",
        "GISCODE پست توزیع",
        "GISCODE فیدر فشار متوسط"
    ]
    if col in data.columns
]

if network_categorical_cols:

    network_missing_count = (
        data[network_categorical_cols]
        .isna()
        .sum(axis=1)
    )

    network_total = len(network_categorical_cols)

    pattern_summary = pd.DataFrame({
        "missing_network_fields": network_missing_count
    })

    pattern_summary["rows"] = 1

    pattern_summary = (
        pattern_summary
        .groupby("missing_network_fields")
        .agg(rows=("rows", "sum"))
        .reset_index()
    )

    pattern_summary["percent"] = (
        pattern_summary["rows"] /
        len(data) * 100
    )

    pattern_summary["interpretation"] = (
        pattern_summary["missing_network_fields"]
        .map(
            lambda x:
            "Complete network information"
            if x == 0
            else
            "Partial network information"
            if x < network_total
            else
            "All network fields missing"
        )
    )

    print("\n" + "=" * 75)
    print("5. NETWORK INFORMATION COMPLETENESS")
    print("=" * 75)

    print(
        f"\nNetwork fields examined: {network_total}"
    )

    display(
        pattern_summary.style.format({
            "percent": "{:.2f}%"
        })
    )


# ------------------------------------------------------------
# 7. Source-file missingness
# ------------------------------------------------------------
# Useful for detecting whether missing values originate from
# particular Excel files rather than the underlying events.

if "source_file" in data.columns:

    print("\n" + "=" * 75)
    print("6. MISSINGNESS BY SOURCE FILE")
    print("=" * 75)

    source_missing = []

    for source, group in data.groupby(
        "source_file",
        dropna=False
    ):

        for col in categorical_cols:

            source_missing.append({
                "source_file": source,
                "column": col,
                "rows": len(group),
                "missing": group[col].isna().sum(),
                "missing_percent":
                    group[col].isna().mean() * 100
            })

    source_missing = pd.DataFrame(source_missing)

    display(
        source_missing
        .sort_values(
            ["column", "missing_percent"],
            ascending=[True, False]
        )
        .style.format({
            "missing_percent": "{:.2f}%"
        })
    )


# ------------------------------------------------------------
# 8. Simple diagnostic classification
# ------------------------------------------------------------
# This is NOT a final decision.
# It flags patterns that deserve investigation.

diagnostic = missing_summary.copy()

diagnostic["diagnostic_flag"] = np.select(
    [
        diagnostic["missing_percent"] == 0,
        diagnostic["missing_percent"] >= 80,
        diagnostic["missing_percent"] > 0
    ],
    [
        "Complete",
        "Highly systematic missingness",
        "Partial missingness"
    ],
    default="Check"
)

print("\n" + "=" * 75)
print("7. INITIAL MISSINGNESS DIAGNOSTIC")
print("=" * 75)

display(
    diagnostic[
        [
            "column",
            "missing_count",
            "missing_percent",
            "unique_observed",
            "diagnostic_flag"
        ]
    ].style.format({
        "missing_percent": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 9. Final interpretation guide
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("INTERPRETATION GUIDE")
print("=" * 75)

print("""
• Complete:
  No missing values; no missing-value treatment is required.

• Partial missingness:
  The feature exists for some events but not others.
  Investigate whether the missingness depends on feeder,
  outage type, date, or another operational condition.

• Highly systematic missingness:
  Missing values occur in a large and potentially structured
  portion of the data. This may indicate that the field is
  conditionally recorded rather than randomly missing.

• Co-missingness:
  If several network fields are missing on exactly the same
  rows, they may represent one missing network-information
  block rather than independent missing values.

IMPORTANT:
This audit does not automatically label a value as
"structurally missing" or "accidentally missing".
It identifies the patterns needed to make that decision
before encoding or imputation.
""")

ORIGINAL DATASET — CATEGORICAL MISSINGNESS AUDIT

Dataset shape: (2764, 55)
Categorical/object columns: 28

Categorical columns:
  • روز هفته
  • معاونت
  • منطقه اصلی
  • تاریخ اطلاع
  • ساعت اطلاع
  • پست فوق توزیع 
  • فیدر فشار متوسط
  • پست توزیع
  • کد پست توزیع
  • آدرس
  •  سطح ولتاژ
  • گروه قطع
  • علت قطع
  • اشکال
  • علت اشکال
  •  جداکننده
  • نوع قطعی
  • وضعیت محل
  • تکی یا کلی به تشخیص اپراتور
  • زون 
  • شرح حادثه
  • زمان قطع
  • زمان وصل
  • كاربري كه ثبت كرده
  • نوع پست
  • گزارش دهنده
  • کاربر ثبت کننده
  • GISCODE پست توزیع

1. OVERALL CATEGORICAL MISSINGNESS


,column,missing_count,missing_percent,observed_count,unique_observed
0,علت اشکال,1762,63.75%,1002,54
1,اشکال,1575,56.98%,1189,116
2,زون,1346,48.70%,1418,224
3,شرح حادثه,1189,43.02%,1575,1467
4,گزارش دهنده,423,15.30%,2341,10
5,کد پست توزیع,106,3.84%,2658,1
6,نوع پست,106,3.84%,2658,6
7,GISCODE پست توزیع,106,3.84%,2658,79
8,پست توزیع,106,3.84%,2658,79
9,آدرس,94,3.40%,2670,2187



2. MISSINGNESS BY MV FEEDER


,feeder,column,rows,missing,missing_percent
71,MV_شريعتي_زمرد,جداکننده,953,16,1.68%
43,MV_شريعتي_حسينيه ارشاد,جداکننده,624,7,1.12%
15,MV_شريعتي_بيات,جداکننده,550,3,0.55%
99,MV_شريعتي_شهرزاد,جداکننده,637,2,0.31%
10,MV_شريعتي_بيات,سطح ولتاژ,550,0,0.00%
38,MV_شريعتي_حسينيه ارشاد,سطح ولتاژ,624,0,0.00%
66,MV_شريعتي_زمرد,سطح ولتاژ,953,0,0.00%
94,MV_شريعتي_شهرزاد,سطح ولتاژ,637,0,0.00%
27,MV_شريعتي_بيات,GISCODE پست توزیع,550,31,5.64%
111,MV_شريعتي_شهرزاد,GISCODE پست توزیع,637,24,3.77%



3. MISSINGNESS BY OUTAGE TYPE

Outage-type distribution:


...


6. MISSINGNESS BY SOURCE FILE


,source_file,column,rows,missing,missing_percent
379,1401-2.xlsx,جداکننده,116,5,4.31%
351,1401-1.xlsx,جداکننده,225,9,4.00%
323,1400-2.xlsx,جداکننده,77,3,3.90%
407,1402-1.xlsx,جداکننده,196,5,2.55%
43,1395-2.xlsx,جداکننده,69,1,1.45%
183,1398-1.xlsx,جداکننده,279,3,1.08%
127,1397-1.xlsx,جداکننده,144,1,0.69%
239,1399-1.xlsx,جداکننده,188,1,0.53%
15,1395-1.xlsx,جداکننده,117,0,0.00%
71,1396-1.xlsx,جداکننده,111,0,0.00%



7. INITIAL MISSINGNESS DIAGNOSTIC


,column,missing_count,missing_percent,unique_observed,diagnostic_flag
0,علت اشکال,1762,63.75%,54,Partial missingness
1,اشکال,1575,56.98%,116,Partial missingness
2,زون,1346,48.70%,224,Partial missingness
3,شرح حادثه,1189,43.02%,1467,Partial missingness
4,گزارش دهنده,423,15.30%,10,Partial missingness
5,کد پست توزیع,106,3.84%,1,Partial missingness
6,نوع پست,106,3.84%,6,Partial missingness
7,GISCODE پست توزیع,106,3.84%,79,Partial missingness
8,پست توزیع,106,3.84%,79,Partial missingness
9,آدرس,94,3.40%,2187,Partial missingness



INTERPRETATION GUIDE

• Complete:
  No missing values; no missing-value treatment is required.

• Partial missingness:
  The feature exists for some events but not others.
  Investigate whether the missingness depends on feeder,
  outage type, date, or another operational condition.

• Highly systematic missingness:
  Missing values occur in a large and potentially structured
  portion of the data. This may indicate that the field is
  conditionally recorded rather than randomly missing.

• Co-missingness:
  If several network fields are missing on exactly the same
  rows, they may represent one missing network-information
  block rather than independent missing values.

IMPORTANT:
This audit does not automatically label a value as
"structurally missing" or "accidentally missing".
It identifies the patterns needed to make that decision
before encoding or imputation.



## Categorical Missingness Interpretation

The missing categorical values represent **two different mechanisms**:

1. **Structural missingness:** Most missing values occur on the 11,762 no-event days because no outage record exists to provide network attributes. These missing values are therefore expected and should not be treated as data errors.

2. **Informational missingness:** Among actual outage days, some network attributes are still missing. `region` and `voltage_level` are complete, while `distribution_substation`, `distribution_gis`, and `substation_type` have only minor missingness. However, `lv_feeder` and especially `zone` have substantial missingness and require further investigation.

**Conclusion:** Missing categorical values should **not be treated uniformly or blindly imputed**. Structural missingness should be preserved as part of the data structure, while event-day missingness should be investigated and handled according to the meaning and reliability of each feature before encoding.

In [8]:
# ============================================================
# ORIGINAL DATA: CATEGORICAL MISSINGNESS & RECOVERABILITY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 75)
print("ORIGINAL DATA — CATEGORICAL MISSINGNESS & RECOVERABILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Work on the ORIGINAL event-level dataset
# ------------------------------------------------------------

raw = data.copy()

# Remove accidental leading/trailing spaces from column names
raw.columns = raw.columns.str.strip()

print("\nOriginal dataset shape:", raw.shape)
print("Original event records:", len(raw))


# ------------------------------------------------------------
# 2. Define target categorical features
# ------------------------------------------------------------

targets = [
    "زون",
    "فیدر فشار ضعیف"
]

# Network fields that may help explain/recover missing values
network_fields = [
    "فیدر فشار متوسط",
    "کد فیدر فشار متوسط",
    "GISCODE فیدر فشار متوسط",
    "پست توزیع",
    "GISCODE پست توزیع",
    "منطقه اصلی",
    "سطح ولتاژ",
    "نوع پست"
]

# Keep only fields that actually exist
network_fields = [
    c for c in network_fields
    if c in raw.columns
]


# ------------------------------------------------------------
# 3. Basic missingness of target variables
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("1. TARGET MISSINGNESS")
print("=" * 75)

missing_summary = pd.DataFrame({
    "feature": targets,
    "missing": [raw[c].isna().sum() for c in targets],
    "missing_percent": [
        raw[c].isna().mean() * 100
        for c in targets
    ],
    "observed": [raw[c].notna().sum() for c in targets],
    "unique_observed": [
        raw[c].nunique()
        for c in targets
    ]
})

display(missing_summary.round(2))


# ------------------------------------------------------------
# 4. Co-missingness analysis
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("2. CO-MISSINGNESS OF NETWORK FEATURES")
print("=" * 75)

for target in targets:

    print(f"\n--- Missing {target} ---")

    missing_target = raw[target].isna()

    rows = []

    for col in network_fields:

        rows.append({
            "network_feature": col,
            "missing_when_target_missing":
                raw.loc[missing_target, col].isna().mean() * 100,
            "missing_when_target_observed":
                raw.loc[~missing_target, col].isna().mean() * 100,
            "observed_values_when_target_missing":
                raw.loc[missing_target, col].notna().sum()
        })

    display(
        pd.DataFrame(rows)
        .sort_values(
            "missing_when_target_missing",
            ascending=False
        )
        .round(2)
    )


# ------------------------------------------------------------
# 5. Missingness by MV feeder
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("3. MISSINGNESS BY MV FEEDER")
print("=" * 75)

feeder_group = [
    c for c in [
        "فیدر فشار متوسط",
        "کد فیدر فشار متوسط",
        "GISCODE فیدر فشار متوسط"
    ]
    if c in raw.columns
]

for target in targets:

    print(f"\n--- {target} ---")

    feeder_missing = (
        raw.groupby(feeder_group, dropna=False)[target]
        .agg(
            total_records="size",
            missing="count"
        )
    )

    # Convert count of observed values into missing count
    feeder_missing["observed"] = feeder_missing["missing"]
    feeder_missing["missing"] = (
        feeder_missing["total_records"]
        - feeder_missing["observed"]
    )

    feeder_missing["missing_percent"] = (
        feeder_missing["missing"]
        / feeder_missing["total_records"]
        * 100
    )

    display(
        feeder_missing
        .sort_values("missing_percent", ascending=False)
        .round(2)
    )


# ------------------------------------------------------------
# 6. Missingness by source file
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("4. MISSINGNESS BY SOURCE FILE")
print("=" * 75)

if "source_file" in raw.columns:

    for target in targets:

        print(f"\n--- {target} ---")

        file_missing = (
            raw.groupby("source_file")[target]
            .agg(
                total_records="size",
                observed="count"
            )
        )

        file_missing["missing"] = (
            file_missing["total_records"]
            - file_missing["observed"]
        )

        file_missing["missing_percent"] = (
            file_missing["missing"]
            / file_missing["total_records"]
            * 100
        )

        display(
            file_missing
            .sort_values("missing_percent", ascending=False)
            .round(2)
        )


# ------------------------------------------------------------
# 7. Deterministic recoverability test
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("5. DETERMINISTIC RECOVERABILITY")
print("=" * 75)

candidate_keys = [
    ["فیدر فشار متوسط"],
    ["کد فیدر فشار متوسط"],
    ["GISCODE فیدر فشار متوسط"],
    ["پست توزیع"],
    ["GISCODE پست توزیع"],
    ["فیدر فشار متوسط", "پست توزیع"],
    ["فیدر فشار متوسط", "سطح ولتاژ"],
    ["GISCODE فیدر فشار متوسط", "پست توزیع"],
    ["GISCODE فیدر فشار متوسط", "GISCODE پست توزیع"],
]

candidate_keys = [
    key for key in candidate_keys
    if all(c in raw.columns for c in key)
]


def recoverability_test(df, target, key):

    # Only rows where target is known
    observed = df[df[target].notna()].copy()

    if observed.empty:
        return None

    # Number of different target values per key
    mapping = (
        observed
        .groupby(key, dropna=False)[target]
        .nunique()
    )

    deterministic = (mapping == 1).sum()
    ambiguous = (mapping > 1).sum()

    # Missing target rows
    missing_rows = df[df[target].isna()].copy()

    if missing_rows.empty:
        recoverable = 0
    else:

        # Create deterministic mapping
        mapping_table = (
            observed
            .groupby(key, dropna=False)[target]
            .agg(
                lambda x:
                x.dropna().iloc[0]
                if x.dropna().nunique() == 1
                else np.nan
            )
            .reset_index()
        )

        # Rename target to avoid collision
        mapping_table = mapping_table.rename(
            columns={target: "_recovered_value"}
        )

        check = missing_rows.merge(
            mapping_table,
            on=key,
            how="left"
        )

        recoverable = check["_recovered_value"].notna().sum()

    total_missing = len(missing_rows)

    return {
        "target": target,
        "key": " + ".join(key),
        "unique_keys": len(mapping),
        "deterministic_keys": deterministic,
        "ambiguous_keys": ambiguous,
        "missing_rows": total_missing,
        "recoverable_rows": recoverable,
        "recovery_percent": (
            recoverable / total_missing * 100
            if total_missing > 0
            else 0
        )
    }


results = []

for target in targets:

    for key in candidate_keys:

        result = recoverability_test(
            raw,
            target,
            key
        )

        if result is not None:
            results.append(result)


recoverability = pd.DataFrame(results)

display(
    recoverability
    .sort_values(
        ["target", "recovery_percent"],
        ascending=[True, False]
    )
    .round(2)
)


# ------------------------------------------------------------
# 8. Target relationships
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("6. NETWORK RELATIONSHIP STABILITY")
print("=" * 75)

relationship_pairs = [
    ("GISCODE پست توزیع", "فیدر فشار ضعیف"),
    ("پست توزیع", "فیدر فشار ضعیف"),
    ("GISCODE فیدر فشار متوسط", "زون"),
    ("فیدر فشار متوسط", "زون"),
    ("GISCODE پست توزیع", "زون"),
    ("پست توزیع", "زون"),
]

for source, target in relationship_pairs:

    if source not in raw.columns or target not in raw.columns:
        continue

    temp = raw[
        raw[source].notna() &
        raw[target].notna()
    ]

    if temp.empty:
        continue

    relationship = (
        temp.groupby(source)[target]
        .nunique()
    )

    deterministic_rate = (
        (relationship == 1).mean() * 100
    )

    print(f"\n{source}  →  {target}")
    print("Source values:", len(relationship))
    print("Deterministic:", (relationship == 1).sum())
    print("Ambiguous:", (relationship > 1).sum())
    print("Deterministic rate:", round(deterministic_rate, 2), "%")


# ------------------------------------------------------------
# 9. Final interpretation indicators
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("7. DIAGNOSTIC INDICATORS")
print("=" * 75)

for target in targets:

    missing_mask = raw[target].isna()

    print(f"\n{target}")
    print("-" * 50)

    print(
        "Missing:",
        missing_mask.sum(),
        f"({missing_mask.mean()*100:.2f}%)"
    )

    # Check whether MV feeder is available when target is missing
    feeder_available = (
        raw.loc[missing_mask, "GISCODE فیدر فشار متوسط"]
        .notna()
        .mean() * 100
    )

    print(
        "MV feeder GIS available among missing:",
        f"{feeder_available:.2f}%"
    )

    # Check whether distribution GIS is available
    if "GISCODE پست توزیع" in raw.columns:

        dist_available = (
            raw.loc[missing_mask, "GISCODE پست توزیع"]
            .notna()
            .mean() * 100
        )

        print(
            "Distribution GIS available among missing:",
            f"{dist_available:.2f}%"
        )


print("\n" + "=" * 75)
print("DIAGNOSTIC COMPLETE")
print("=" * 75)

ORIGINAL DATA — CATEGORICAL MISSINGNESS & RECOVERABILITY

Original dataset shape: (2764, 55)
Original event records: 2764

1. TARGET MISSINGNESS


,feature,missing,missing_percent,observed,unique_observed
0,زون,1346,48.70,1418,224
1,فیدر فشار ضعیف,794,28.73,1970,19



2. CO-MISSINGNESS OF NETWORK FEATURES

--- Missing زون ---


,network_feature,missing_when_target_missing,missing_when_target_observed,observed_values_when_target_missing
3,پست توزیع,7.73,0.14,1242
4,GISCODE پست توزیع,7.73,0.14,1242
7,نوع پست,7.73,0.14,1242
0,فیدر فشار متوسط,0.00,0.00,1346
2,GISCODE فیدر فشار متوسط,0.00,0.00,1346
1,کد فیدر فشار متوسط,0.00,0.00,1346
5,منطقه اصلی,0.00,0.00,1346
6,سطح ولتاژ,0.00,0.00,1346



--- Missing فیدر فشار ضعیف ---


...


--- فیدر فشار ضعیف ---


,total_records,observed,missing,missing_percent
source_file,,,,
1399-2.xlsx,92,26,66,71.74
1400-2.xlsx,77,30,47,61.04
1401-2.xlsx,116,68,48,41.38
1395-1.xlsx,117,69,48,41.03
1397-2.xlsx,98,59,39,39.80
1402-2.xlsx,104,67,37,35.58
1401-1.xlsx,225,150,75,33.33
1403-2.xlsx,76,52,24,31.58
1400-1.xlsx,202,145,57,28.22



5. DETERMINISTIC RECOVERABILITY


,target,key,unique_keys,deterministic_keys,ambiguous_keys,missing_rows,recoverable_rows,recovery_percent
3,زون,پست توزیع,51,8,43,1346,22,1.63
4,زون,GISCODE پست توزیع,51,8,43,1346,22,1.63
5,زون,فیدر فشار متوسط + پست توزیع,51,8,43,1346,22,1.63
7,زون,GISCODE فیدر فشار متوسط + پست توزیع,51,8,43,1346,22,1.63
8,زون,GISCODE فیدر فشار متوسط + GISCODE پست توزیع,51,8,43,1346,22,1.63
0,زون,فیدر فشار متوسط,4,0,4,1346,0,0.00
1,زون,کد فیدر فشار متوسط,4,0,4,1346,0,0.00
2,زون,GISCODE فیدر فشار متوسط,4,0,4,1346,0,0.00
6,زون,فیدر فشار متوسط + سطح ولتاژ,8,0,8,1346,0,0.00
12,فیدر فشار ضعیف,پست توزیع,67,12,55,794,109,13.73



6. NETWORK RELATIONSHIP STABILITY

GISCODE پست توزیع  →  فیدر فشار ضعیف
Source values: 66
Deterministic: 11
Ambiguous: 55
Deterministic rate: 16.67 %

پست توزیع  →  فیدر فشار ضعیف
Source values: 66
Deterministic: 11
Ambiguous: 55
Deterministic rate: 16.67 %

GISCODE فیدر فشار متوسط  →  زون
Source values: 4
Deterministic: 0
Ambiguous: 4
Deterministic rate: 0.0 %

فیدر فشار متوسط  →  زون
Source values: 4
Deterministic: 0
Ambiguous: 4
Deterministic rate: 0.0 %

GISCODE پست توزیع  →  زون
Source values: 50
Deterministic: 8
Ambiguous: 42
Deterministic rate: 16.0 %

پست توزیع  →  زون
Source values: 50
Deterministic: 8
Ambiguous: 42
Deterministic rate: 16.0 %

7. DIAGNOSTIC INDICATORS

زون
--------------------------------------------------
Missing: 1346 (48.70%)
MV feeder GIS available among missing: 100.00%
Distribution GIS available among missing: 92.27%

فیدر فشار ضعیف
--------------------------------------------------
Missing: 794 (28.73%)
MV feeder GIS available among missing: 100.00%
Di

## Investigating Temporal and Source-File Stability of `zone` and `lv_feeder`

Before deciding how to encode `zone` and `lv_feeder`, we first need to determine whether their meaning and distribution are stable across years and source files.

This analysis examines:

1. Missingness and the number of observed categories across source files.
2. Changes in category distributions over time.
3. Categories that appear or disappear in particular years/files.
4. Whether the same network asset consistently maps to the same `zone` or `lv_feeder`.
5. Whether apparent ambiguity is caused by genuine network variation or by changes across years/source files.
6. Whether the representation/format of the categorical values itself changes over time.

The objective is to distinguish between:
- **structural missingness** caused by the data collection process,
- **legitimate temporal/network changes**, and
- **inconsistent categorical representation**.

This evidence will be used to decide whether `zone` and `lv_feeder` should be retained as categorical predictors, reconstructed from network identifiers, or excluded from modeling.

In [10]:
# ============================================================
# TEMPORAL & SOURCE-FILE STABILITY AUDIT
# zone + lv_feeder
# ============================================================

import pandas as pd
import numpy as np
import re

print("=" * 75)
print("TEMPORAL & SOURCE-FILE STABILITY AUDIT")
print("zone + lv_feeder")
print("=" * 75)


# ------------------------------------------------------------
# 1. Clean column names
# ------------------------------------------------------------

# Important:
# The original Excel data contains names such as "زون " with
# trailing whitespace and " سطح ولتاژ" with leading whitespace.

data.columns = (
    data.columns
    .astype(str)
    .str.strip()
)

print("\nColumn names cleaned successfully.")


# ------------------------------------------------------------
# 2. Define original-data feature names
# ------------------------------------------------------------

target_cols = [
    "زون",
    "فیدر فشار ضعیف"
]

required_cols = [
    "source_file",
    "تاریخ اطلاع",
    "فیدر فشار متوسط",
    "کد فیدر فشار متوسط",
    "GISCODE فیدر فشار متوسط",
    "پست توزیع",
    "GISCODE پست توزیع",
    "نوع پست",
    " سطح ولتاژ".strip(),
    "زون",
    "فیدر فشار ضعیف"
]

missing_columns = [
    col for col in required_cols
    if col not in data.columns
]

if missing_columns:
    raise KeyError(
        "Missing required columns:\n"
        + "\n".join(missing_columns)
    )


# ------------------------------------------------------------
# 3. Create analysis copy
# ------------------------------------------------------------

audit = data.copy()


# ------------------------------------------------------------
# 4. Extract Jalali year from source filename
# ------------------------------------------------------------

# Examples:
# 1395-1.xlsx
# 1395-2.xlsx
# 1404-1-.xlsx

audit["source_year"] = (
    audit["source_file"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
)

audit["source_year"] = pd.to_numeric(
    audit["source_year"],
    errors="coerce"
)


# Source period (first / second half)
audit["source_period"] = (
    audit["source_file"]
    .astype(str)
    .str.extract(r"\d{4}-(\d+)")[0]
)

audit["source_period"] = pd.to_numeric(
    audit["source_period"],
    errors="coerce"
)


print("\nSource years:")
print(
    sorted(
        audit["source_year"]
        .dropna()
        .unique()
    )
)


# ============================================================
# 5. OVERALL TEMPORAL MISSINGNESS
# ============================================================

print("\n" + "=" * 75)
print("1. MISSINGNESS BY SOURCE YEAR")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    yearly = (
        audit
        .groupby("source_year", dropna=False)
        .agg(
            total_records=(target, "size"),
            observed=(target, "count"),
            missing=(target, lambda x: x.isna().sum()),
            unique_observed=(target, "nunique")
        )
        .reset_index()
    )

    yearly["missing_percent"] = (
        yearly["missing"]
        / yearly["total_records"]
        * 100
    )

    display(
        yearly[
            [
                "source_year",
                "total_records",
                "observed",
                "missing",
                "missing_percent",
                "unique_observed"
            ]
        ].round(2)
    )


# ============================================================
# 6. MISSINGNESS BY SOURCE FILE
# ============================================================

print("\n" + "=" * 75)
print("2. MISSINGNESS BY SOURCE FILE")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    file_summary = (
        audit
        .groupby("source_file", dropna=False)
        .agg(
            total_records=(target, "size"),
            observed=(target, "count"),
            missing=(target, lambda x: x.isna().sum()),
            unique_observed=(target, "nunique")
        )
        .reset_index()
    )

    file_summary["missing_percent"] = (
        file_summary["missing"]
        / file_summary["total_records"]
        * 100
    )

    file_summary["source_year"] = (
        file_summary["source_file"]
        .str.extract(r"(\d{4})")[0]
        .astype(int)
    )

    file_summary["source_period"] = (
        file_summary["source_file"]
        .str.extract(r"\d{4}-(\d+)")[0]
        .astype(int)
    )

    file_summary = file_summary.sort_values(
        ["source_year", "source_period"]
    )

    display(
        file_summary[
            [
                "source_year",
                "source_period",
                "source_file",
                "total_records",
                "observed",
                "missing",
                "missing_percent",
                "unique_observed"
            ]
        ].round(2)
    )


# ============================================================
# 7. CATEGORY DISTRIBUTION BY YEAR
# ============================================================

print("\n" + "=" * 75)
print("3. CATEGORY DISTRIBUTION BY SOURCE YEAR")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    observed = audit[
        audit[target].notna()
    ].copy()

    distribution = (
        observed
        .groupby(["source_year", target])
        .size()
        .reset_index(name="count")
    )

    distribution["year_total"] = (
        distribution
        .groupby("source_year")["count"]
        .transform("sum")
    )

    distribution["percent"] = (
        distribution["count"]
        / distribution["year_total"]
        * 100
    )

    display(
        distribution
        .sort_values(
            ["source_year", "count"],
            ascending=[True, False]
        )
        .round(2)
    )


# ============================================================
# 8. CATEGORY APPEARANCE / DISAPPEARANCE ACROSS YEARS
# ============================================================

print("\n" + "=" * 75)
print("4. CATEGORY TEMPORAL PRESENCE")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    presence = (
        audit[
            audit[target].notna()
        ]
        .groupby(target)["source_year"]
        .agg(
            years_present=lambda x: sorted(x.unique()),
            number_of_years=lambda x: x.nunique()
        )
        .reset_index()
    )

    total_years = audit["source_year"].nunique()

    presence["coverage_percent"] = (
        presence["number_of_years"]
        / total_years
        * 100
    )

    display(
        presence
        .sort_values(
            ["number_of_years", target],
            ascending=[False, True]
        )
        .round(2)
    )


# ============================================================
# 9. CATEGORY STABILITY:
#    Does each category stay associated with the same MV feeder?
# ============================================================

print("\n" + "=" * 75)
print("5. CATEGORY ↔ MV FEEDER STABILITY")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    temp = audit[
        audit[target].notna()
        & audit["فیدر فشار متوسط"].notna()
    ].copy()

    stability = (
        temp
        .groupby(target)["فیدر فشار متوسط"]
        .nunique()
        .reset_index(
            name="number_of_mv_feeders"
        )
    )

    stability["stable_single_mv_feeder"] = (
        stability["number_of_mv_feeders"] == 1
    )

    display(
        stability
        .sort_values(
            "number_of_mv_feeders",
            ascending=False
        )
    )

    print(
        "Categories mapped to exactly one MV feeder:",
        (
            stability["stable_single_mv_feeder"]
            .sum()
        ),
        "/",
        len(stability)
    )


# ============================================================
# 10. CATEGORY ↔ DISTRIBUTION SUBSTATION STABILITY
# ============================================================

print("\n" + "=" * 75)
print("6. CATEGORY ↔ DISTRIBUTION SUBSTATION STABILITY")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    temp = audit[
        audit[target].notna()
        & audit["GISCODE پست توزیع"].notna()
    ].copy()

    stability = (
        temp
        .groupby(target)["GISCODE پست توزیع"]
        .nunique()
        .reset_index(
            name="number_of_distribution_assets"
        )
    )

    stability["stable_single_asset"] = (
        stability["number_of_distribution_assets"] == 1
    )

    display(
        stability
        .sort_values(
            "number_of_distribution_assets",
            ascending=False
        )
    )

    print(
        "Categories mapped to exactly one distribution asset:",
        stability["stable_single_asset"].sum(),
        "/",
        len(stability)
    )


# ============================================================
# 11. SOURCE-FILE CATEGORY OVERLAP
# ============================================================

print("\n" + "=" * 75)
print("7. CATEGORY OVERLAP BETWEEN SOURCE FILES")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    category_sets = {}

    for file_name, group in audit.groupby("source_file"):

        values = set(
            group[target]
            .dropna()
            .astype(str)
            .unique()
        )

        category_sets[file_name] = values

    files = sorted(category_sets.keys())

    overlap_results = []

    for i in range(len(files)):

        for j in range(i + 1, len(files)):

            a = category_sets[files[i]]
            b = category_sets[files[j]]

            if len(a) == 0 or len(b) == 0:
                continue

            intersection = len(a & b)
            union = len(a | b)

            jaccard = (
                intersection / union
                if union > 0
                else np.nan
            )

            overlap_results.append({
                "file_A": files[i],
                "file_B": files[j],
                "categories_A": len(a),
                "categories_B": len(b),
                "shared_categories": intersection,
                "jaccard_similarity": jaccard
            })

    overlap_df = pd.DataFrame(overlap_results)

    display(
        overlap_df
        .sort_values(
            "jaccard_similarity",
            ascending=True
        )
        .round(3)
    )


# ============================================================
# 12. CATEGORY DISTRIBUTION CONCENTRATION
# ============================================================

print("\n" + "=" * 75)
print("8. DISTRIBUTION CONCENTRATION BY SOURCE YEAR")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    rows = []

    for year, group in audit.groupby("source_year"):

        values = (
            group[target]
            .dropna()
        )

        if len(values) == 0:
            continue

        proportions = (
            values
            .value_counts(normalize=True)
        )

        rows.append({
            "source_year": year,
            "observed_records": len(values),
            "unique_categories": values.nunique(),
            "dominant_category": proportions.index[0],
            "dominant_category_percent": proportions.iloc[0] * 100,
            "top_3_percent": proportions.iloc[:3].sum() * 100
        })

    concentration = pd.DataFrame(rows)

    display(
        concentration.round(2)
    )


# ============================================================
# 13. YEAR × MV FEEDER × TARGET STABILITY
# ============================================================

print("\n" + "=" * 75)
print("9. TEMPORAL STABILITY WITHIN MV FEEDERS")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    temp = audit[
        audit[target].notna()
    ].copy()

    stability = (
        temp
        .groupby(
            [
                "source_year",
                "فیدر فشار متوسط"
            ]
        )[target]
        .nunique()
        .reset_index(
            name="unique_target_values"
        )
    )

    stability["stable"] = (
        stability["unique_target_values"] == 1
    )

    display(
        stability
        .sort_values(
            [
                "source_year",
                "فیدر فشار متوسط"
            ]
        )
    )

    print(
        "Stable feeder-year combinations:",
        stability["stable"].sum(),
        "/",
        len(stability)
    )


# ============================================================
# 14. FINAL AUTOMATIC FLAGS
# ============================================================

print("\n" + "=" * 75)
print("10. DECISION-ORIENTED DIAGNOSTIC FLAGS")
print("=" * 75)


for target in target_cols:

    print(f"\n--- {target} ---")

    missing_rate = (
        audit[target].isna().mean() * 100
    )

    n_categories = (
        audit[target].nunique()
    )

    years_with_data = (
        audit.loc[
            audit[target].notna(),
            "source_year"
        ].nunique()
    )

    total_years = audit["source_year"].nunique()

    print(
        f"Overall missingness: {missing_rate:.2f}%"
    )

    print(
        f"Observed categories: {n_categories}"
    )

    print(
        f"Years with observations: "
        f"{years_with_data}/{total_years}"
    )

    if missing_rate > 30:
        print(
            "⚠ HIGH MISSINGNESS — investigate recording mechanism."
        )
    else:
        print(
            "✓ Missingness is relatively limited."
        )

    if years_with_data < total_years:
        print(
            "⚠ CATEGORY INFORMATION IS NOT PRESENT "
            "THROUGHOUT THE ENTIRE PERIOD."
        )
    else:
        print(
            "✓ Feature appears across the full study period."
        )


print("\n" + "=" * 75)
print("TEMPORAL & SOURCE-FILE AUDIT COMPLETE")
print("=" * 75)

TEMPORAL & SOURCE-FILE STABILITY AUDIT
zone + lv_feeder

Column names cleaned successfully.

Source years:
[np.int64(1395), np.int64(1396), np.int64(1397), np.int64(1398), np.int64(1399), np.int64(1400), np.int64(1401), np.int64(1402), np.int64(1403), np.int64(1404)]

1. MISSINGNESS BY SOURCE YEAR

--- زون ---


,source_year,total_records,observed,missing,missing_percent,unique_observed
0,1395,186,0,186,100.00,0
1,1396,207,0,207,100.00,0
2,1397,242,0,242,100.00,0
3,1398,363,0,363,100.00,0
4,1399,280,88,192,68.57,31
5,1400,279,242,37,13.26,93
6,1401,341,291,50,14.66,107
7,1402,300,280,20,6.67,18
8,1403,389,361,28,7.20,25
9,1404,177,156,21,11.86,15



--- فیدر فشار ضعیف ---


,source_year,total_records,observed,missing,missing_percent,unique_observed
0,1395,186,120,66,35.48,13
1,1396,207,153,54,26.09,11
2,1397,242,173,69,28.51,12
3,1398,363,312,51,14.05,12
4,1399,280,165,115,41.07,11
5,1400,279,175,104,37.28,14
6,1401,341,218,123,36.07,14
7,1402,300,217,83,27.67,13
8,1403,389,301,88,22.62,13
9,1404,177,136,41,23.16,13



2. MISSINGNESS BY SOURCE FILE

--- زون ---


...


9. TEMPORAL STABILITY WITHIN MV FEEDERS

--- زون ---


,source_year,فیدر فشار متوسط,unique_target_values,stable
0,1399,MV_شريعتي_بيات,7,False
1,1399,MV_شريعتي_حسينيه ارشاد,8,False
2,1399,MV_شريعتي_زمرد,16,False
3,1399,MV_شريعتي_شهرزاد,9,False
4,1400,MV_شريعتي_بيات,18,False
5,1400,MV_شريعتي_حسينيه ارشاد,32,False
6,1400,MV_شريعتي_زمرد,34,False
7,1400,MV_شريعتي_شهرزاد,26,False
8,1401,MV_شريعتي_بيات,27,False
9,1401,MV_شريعتي_حسينيه ارشاد,28,False


Stable feeder-year combinations: 0 / 24

--- فیدر فشار ضعیف ---


,source_year,فیدر فشار متوسط,unique_target_values,stable
0,1395,MV_شريعتي_بيات,8,False
1,1395,MV_شريعتي_حسينيه ارشاد,9,False
2,1395,MV_شريعتي_زمرد,10,False
3,1395,MV_شريعتي_شهرزاد,7,False
4,1396,MV_شريعتي_بيات,8,False
5,1396,MV_شريعتي_حسينيه ارشاد,10,False
6,1396,MV_شريعتي_زمرد,11,False
7,1396,MV_شريعتي_شهرزاد,8,False
8,1397,MV_شريعتي_بيات,12,False
9,1397,MV_شريعتي_حسينيه ارشاد,9,False


Stable feeder-year combinations: 0 / 40

10. DECISION-ORIENTED DIAGNOSTIC FLAGS

--- زون ---
Overall missingness: 48.70%
Observed categories: 224
Years with observations: 6/10
⚠ HIGH MISSINGNESS — investigate recording mechanism.
⚠ CATEGORY INFORMATION IS NOT PRESENT THROUGHOUT THE ENTIRE PERIOD.

--- فیدر فشار ضعیف ---
Overall missingness: 28.73%
Observed categories: 19
Years with observations: 10/10
✓ Missingness is relatively limited.
✓ Feature appears across the full study period.

TEMPORAL & SOURCE-FILE AUDIT COMPLETE


### `zone` → ⚠️ Problematic

`zone` is missing **48.7% overall**, is completely absent from **1395–1398**, and its category distribution changes substantially across years/source files. It also cannot be reliably reconstructed from MV feeder or other network identifiers.

Therefore, `zone` should **not be used as a primary categorical predictor**.

### `lv_feeder` → ✅ Much more usable

`lv_feeder` exists throughout the study period, has only **28.7% missingness**, and its main categories (especially **1–12**) recur across the study period. Its underlying meaning therefore appears considerably more stable.

However, `lv_feeder` requires **cleaning before encoding**. Values such as `9999`, `101782`, and possibly `0` may represent exceptional or administrative codes rather than genuine feeder categories. These values should be investigated first.

### Important: Do Not Blindly Impute

The recoverability analysis showed that the available network identifiers cannot deterministically recover most missing values.

Therefore, imputing missing `zone` or `lv_feeder` values from MV feeder or other network fields could introduce **false network information**.

## Recommended Decision

| Feature | Decision | Reason |
|---|---|---|
| `zone` | ⚠️ Exclude or use cautiously | High and structurally changing missingness |
| `lv_feeder` | ✅ Keep | Present throughout the study period and substantially more stable |
| Missing `lv_feeder` | Keep as `Unknown/Missing` + missingness indicator | Avoid fabricated feeder identities |
| Missing `zone` | Use `Unknown` only if the feature is retained | Missingness itself may contain information |
| One-hot encoding | ❌ Not automatically necessary | 224 `zone` categories would create unnecessary dimensionality |

## Key Conclusion

This is **not simply a missing-value problem**.

For `zone`, the evidence strongly suggests a **data-collection or recording regime change over time**. Therefore, its missingness and changing category structure make it unreliable as a primary categorical predictor.

For `lv_feeder`, the feature appears to represent a legitimate and relatively stable **network identifier**, although some exceptional values must be investigated before encoding.

### Final Check Before Encoding

The next and final audit should focus specifically on `lv_feeder` values:

- `0`
- `1–21`
- `9999`
- `101782`
- missing values

The goal is to determine whether these exceptional values are **genuine feeder identifiers or administrative/data-entry codes**.

Only after this check should the final encoding strategy for `lv_feeder` be selected.

In [12]:
# ============================================================
# 6. STABILITY OF LV FEEDER VALUES ACROSS YEARS / SOURCE FILES
# ============================================================

# Clean accidental whitespace from column names
data.columns = data.columns.str.strip()

lv_col = "فیدر فشار ضعیف"
date_col = "تاریخ اطلاع"
source_col = "source_file"

print("\n" + "=" * 80)
print("LV FEEDER STABILITY ACROSS YEARS / SOURCE FILES")
print("=" * 80)

# ------------------------------------------------------------
# Convert Jalali year from the original date
# ------------------------------------------------------------

data["jalali_year"] = (
    data[date_col]
    .astype(str)
    .str.extract(r"^(\d{4})")[0]
)

# Keep only genuine-looking feeder values: 1–21
valid_lv = data[
    data[lv_col].notna() &
    data[lv_col].between(1, 21)
].copy()

# ------------------------------------------------------------
# A. Overall distribution of LV feeder values
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("A. OVERALL DISTRIBUTION OF VALID LV FEEDERS (1–21)")
print("=" * 80)

overall = (
    valid_lv[lv_col]
    .value_counts()
    .sort_index()
    .rename_axis("lv_feeder")
    .reset_index(name="records")
)

overall["percent"] = (
    overall["records"] / len(valid_lv) * 100
)

display(overall.round(2))

# ------------------------------------------------------------
# B. Presence across source files / years
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("B. LV FEEDER PRESENCE ACROSS SOURCE FILES")
print("=" * 80)

presence = pd.crosstab(
    valid_lv[lv_col],
    valid_lv[source_col]
)

display(presence)

# ------------------------------------------------------------
# C. Number of different LV feeders in each source file
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("C. NUMBER OF VALID LV FEEDERS PER SOURCE FILE")
print("=" * 80)

file_summary = (
    valid_lv
    .groupby(source_col)[lv_col]
    .agg(
        records="size",
        unique_lv_feeders="nunique"
    )
    .reset_index()
)

display(file_summary)

# ------------------------------------------------------------
# D. LV feeder distribution by Jalali year
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("D. LV FEEDER DISTRIBUTION BY YEAR")
print("=" * 80)

year_distribution = pd.crosstab(
    valid_lv["jalali_year"],
    valid_lv[lv_col],
    normalize="index"
) * 100

display(
    year_distribution
    .round(2)
)

# ------------------------------------------------------------
# E. Check whether individual feeder IDs recur across years
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("E. RECURRENCE OF EACH LV FEEDER ACROSS YEARS")
print("=" * 80)

recurrence = (
    valid_lv
    .groupby(lv_col)["jalali_year"]
    .nunique()
    .reset_index(name="number_of_years")
)

recurrence["appears_in_all_years"] = (
    recurrence["number_of_years"] ==
    valid_lv["jalali_year"].nunique()
)

display(recurrence)

# ------------------------------------------------------------
# F. Check whether exceptional values behave differently
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("F. EXCEPTIONAL VALUES")
print("=" * 80)

exceptional_values = [0, 9999, 101782]

exceptional = data[
    data[lv_col].isin(exceptional_values)
].copy()

if len(exceptional) > 0:

    display(
        exceptional[
            [
                lv_col,
                date_col,
                source_col,
                "فیدر فشار متوسط",
                "کد فیدر فشار متوسط",
                "GISCODE فیدر فشار متوسط",
                "پست توزیع",
                "GISCODE پست توزیع",
                "منطقه اصلی",
                "نوع پست"
            ]
        ]
    )

else:
    print("No exceptional values found.")

# ------------------------------------------------------------
# G. Final compact diagnostic
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL LV FEEDER DIAGNOSTIC")
print("=" * 80)

print(f"Valid LV feeder values (1–21): {len(valid_lv):,}")
print(f"Number of valid feeder IDs: {valid_lv[lv_col].nunique()}")
print(f"Years represented: {valid_lv['jalali_year'].nunique()}")
print(
    f"Exceptional values (0, 9999, 101782): "
    f"{len(exceptional):,}"
)

print("\nAudit complete.")


LV FEEDER STABILITY ACROSS YEARS / SOURCE FILES

A. OVERALL DISTRIBUTION OF VALID LV FEEDERS (1–21)


,lv_feeder,records,percent
0,1.0,304,15.47
1,2.0,368,18.73
2,3.0,280,14.25
3,4.0,263,13.38
4,5.0,174,8.85
5,6.0,183,9.31
6,7.0,84,4.27
7,8.0,149,7.58
8,9.0,63,3.21
9,10.0,67,3.41



B. LV FEEDER PRESENCE ACROSS SOURCE FILES


source_file,1395-1.xlsx,1395-2.xlsx,1396-1.xlsx,1396-2.xlsx,1397-1.xlsx,1397-2.xlsx,1398-1.xlsx,1398-2.xlsx,1399-1.xlsx,1399-2.xlsx,1400-1.xlsx,1400-2.xlsx,1401-1.xlsx,1401-2.xlsx,1402-1.xlsx,1402-2.xlsx,1403-1.xlsx,1403-2.xlsx,1404-1-.xlsx
فیدر فشار ضعیف,,,,,,,,,,,,,,,,,,,
1.0,10,8,9,2,16,8,32,9,11,3,19,13,21,13,25,22,45,22,16
2.0,4,9,15,9,8,5,23,5,22,7,29,7,48,21,40,18,57,3,38
3.0,14,9,7,14,19,4,45,9,15,3,25,0,22,9,20,7,32,6,20
4.0,12,4,11,9,14,6,46,8,22,6,23,3,17,11,12,10,29,4,16
5.0,9,1,13,4,9,7,19,8,13,5,12,1,9,4,15,3,20,3,19
6.0,10,8,7,14,20,7,26,4,18,1,11,3,9,2,6,1,20,4,12
7.0,1,2,3,7,5,7,10,3,11,0,8,0,1,1,9,2,6,4,4
8.0,6,4,5,5,10,10,22,10,20,1,8,0,11,0,13,2,17,1,4
9.0,0,4,7,0,7,1,13,4,3,0,5,1,2,4,1,0,8,2,1



C. NUMBER OF VALID LV FEEDERS PER SOURCE FILE


...


E. RECURRENCE OF EACH LV FEEDER ACROSS YEARS


,فیدر فشار ضعیف,number_of_years,appears_in_all_years
0,1.0,10,True
1,2.0,10,True
2,3.0,10,True
3,4.0,10,True
4,5.0,10,True
5,6.0,10,True
6,7.0,10,True
7,8.0,10,True
8,9.0,10,True
9,10.0,10,True



F. EXCEPTIONAL VALUES


,فیدر فشار ضعیف,تاریخ اطلاع,source_file,فیدر فشار متوسط,کد فیدر فشار متوسط,GISCODE فیدر فشار متوسط,پست توزیع,GISCODE پست توزیع,منطقه اصلی,نوع پست
79,101782.0,1404/06/12,1404-1-.xlsx,MV_شريعتي_شهرزاد,25794052,112280,بهشت 1,E914@GP1307000432FC,پاسداران,زمينى
836,9999.0,1403/05/24,1403-1.xlsx,MV_شريعتي_شهرزاد,25794052,112280,بهشت مريم,E914@00000SB40092,پاسداران,كيوسك
1445,9999.0,1401/11/08,1401-2.xlsx,MV_شريعتي_حسينيه ارشاد,25794042,112260,حسينيه ارشاد,E914@UP13080000C04D,پاسداران,زيرزمينى
1830,0.0,1395/03/05,1395-1.xlsx,MV_شريعتي_زمرد,25794022,112240,زمرد,E914@GP13130000C974,پاسداران,زمينى
2445,9999.0,1402/05/07,1402-1.xlsx,MV_شريعتي_حسينيه ارشاد,25794042,112260,حسينيه ارشاد,E914@UP13080000C04D,پاسداران,زيرزمينى



FINAL LV FEEDER DIAGNOSTIC
Valid LV feeder values (1–21): 1,965
Number of valid feeder IDs: 16
Years represented: 10
Exceptional values (0, 9999, 101782): 5

Audit complete.


## Final Decision: `lv_feeder`

The audit shows that `lv_feeder` is a **stable network identifier** across the 10-year dataset.

- Values **1–15 and 21** are retained as legitimate feeder IDs.
- Values **0, 9999, and 101782** are extremely rare and treated as **missing/anomalous values**.
- Missing values (28.7%) should **not be artificially imputed** because reliable recovery is not supported by the data.
- `lv_feeder` should be treated as a **nominal categorical variable**, not as a numerical feature.

### Recommended Treatment

| Value | Treatment |
|---|---|
| 1–15, 21 | Keep as categorical IDs |
| 0 | Convert to missing |
| 9999 | Convert to missing |
| 101782 | Convert to missing |
| Missing | Keep as missing/Unknown |

**Conclusion:** `lv_feeder` is suitable for modeling and should be retained. For encoding, prefer a **native categorical method such as CatBoost** rather than treating feeder IDs as numerical values or automatically creating many one-hot features.